<a href="https://colab.research.google.com/github/ubermanproductionsva-jpg/helix-rna-folding-engine/blob/main/AOFE_v14_MASTER_SOLVER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ==============================================================================
# CELL 01 — THE AOFE v14.5 CORE KNOWLEDGE ENGINE
# Purpose: Unified Infrastructure + 27 Axiomatic Geometric Primitives
# ==============================================================================

import os, json, yaml, time, random, numpy as np
from pathlib import Path
from collections import deque, Counter
from scipy.ndimage import label, find_objects as scipy_find_objects
from google.colab import drive

# 1. INFRASTRUCTURE: DATA BINDING & DRIVE MOUNT
# ------------------------------------------------------------------------------
drive.mount('/content/drive', force_remount=True)

AOFE_PATHS = {
    "logic_registry": "/content/drive/MyDrive/ARC_PROJECT/logic_registry (4).yaml",
    "training_challenges": "/content/drive/MyDrive/arc stuff/arc keep/3 29/arc_tasks/arc-agi_training_challenges (3).json",
    "test_challenges": "/content/drive/MyDrive/arc stuff/arc keep/3 29/arc-agi_test_challenges (2).json",
}

print("📡 Booting Data Pipes...")
try:
    with open(AOFE_PATHS["training_challenges"], 'r') as f:
        training_challenges = json.load(f)
    with open(AOFE_PATHS["test_challenges"], 'r') as f:
        test_challenges = json.load(f)
    print(f"✅ Data Loaded: {len(training_challenges)} Training | {len(test_challenges)} Test")
except Exception as e:
    print(f"❌ CRITICAL LOAD FAILURE: {e}")

# 2. CORE KNOWLEDGE AXIOMS (THE LAWS OF PHYSICS)
# ------------------------------------------------------------------------------

def get_bg_axiom(grid):
    """Axiom 02: Background Stability."""
    edges = np.concatenate([grid[0,:], grid[-1,:], grid[:,0], grid[:,-1]])
    return Counter(edges).most_common(1)[0][0] if edges.size > 0 else 0

def get_objects_axiom(grid, connectivity=8):
    """Axiom 03/20: Object-Centricity (Connectivity Physics)."""
    bg = get_bg_axiom(grid)
    mask = (grid != bg)
    struct = np.ones((3,3)) if connectivity == 8 else [[0,1,0],[1,1,1],[0,1,0]]
    labeled, num = label(mask, structure=struct)
    slices = scipy_find_objects(labeled)

    results = []
    for i, slc in enumerate(slices):
        if slc is None: continue
        obj_mask = (labeled[slc] == i + 1)
        results.append({
            'grid': np.where(obj_mask, grid[slc], bg),
            'origin': (slc[0].start, slc[1].start),
            'color': Counter(grid[slc][obj_mask]).most_common(1)[0][0]
        })
    return results

def get_topology_axiom(grid):
    """Axiom 18: Enclosure & Hole Detection."""
    bg = get_bg_axiom(grid)
    padded = np.pad(grid == bg, 1, constant_values=True)
    outside = np.zeros_like(padded, dtype=bool)
    q = deque([(0,0)])
    outside[0,0] = True
    while q:
        r, c = q.popleft()
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < padded.shape[0] and 0 <= nc < padded.shape[1] and \
               not outside[nr,nc] and padded[nr,nc]:
                outside[nr,nc] = True
                q.append((nr,nc))
    return (padded == True)[1:-1, 1:-1] & (~outside[1:-1, 1:-1])

# 3. DSL GRAMMAR & SCHEMA SHIELDS
# ------------------------------------------------------------------------------

class Node:
    def eval(self, grid): return np.array(grid)

class OpNode(Node):
    def __init__(self, op_id): self.op_id = op_id
    def eval(self, grid): return OPS.get(self.op_id, lambda g: g)(grid)

# 4. REGISTRY BINDING
# ------------------------------------------------------------------------------

def build_ops_v14_5():
    ops = {"Identity": lambda g: g}

    # Load 101 YAML Skills
    try:
        with open(AOFE_PATHS["logic_registry"], 'r') as f:
            registry = yaml.safe_load(f)
        for entry in registry:
            try:
                local_env = {}
                exec(entry['implementation_logic'], {"np": np, "deque": deque}, local_env)
                if 'op' in local_env:
                    func = local_env['op']
                    ops[entry['entry_id']] = lambda g, f=func: f(g)
            except: continue
    except: print("⚠️ Warning: Skill Registry skipped.")

    # Inject Active Axioms as Tools
    ops["AXIOM_18_FILL"] = lambda g: np.where(get_topology_axiom(g), 1, g)
    ops["AXIOM_12_FLIP"] = lambda g: np.fliplr(np.array(g))

    return ops

OPS = build_ops_v14_5()
print(f"🔥 MEGA-ENGINE LOADED: {len(OPS)} Active Primitives ready for Sensory Processing.")


Mounted at /content/drive
📡 Booting Data Pipes...
✅ Data Loaded: 1000 Training | 240 Test
🔥 MEGA-ENGINE LOADED: 103 Active Primitives ready for Sensory Processing.


In [9]:
# ==============================================================================
# CELL 02 — THE SENSORY & CONSCIENCE LAYER (AOFE v14.5)
# Purpose: Vectorized Context Extraction & Axiomatic Scoring (The Hand & The Eye)
# ==============================================================================

def get_full_context(grid):
    """
    INTEGRATED PERCEPTION: Architect's Vectorization + Axiom 27 Normalization.
    """
    g = np.array(grid)
    bg_color = get_bg_axiom(g)

    # Extract objects with 8-connectivity (Axiom 03)
    # We use the hard-coded math from Cell 01
    raw_objs = get_objects_axiom(g, diag=True)

    context = {
        "grid": g,
        "bg_color": bg_color,
        "objects": [],
        "roles": {"unique_marker": None, "anchor_obj": None}
    }

    for slc in raw_objs:
        obj_data = g[slc]
        coords = np.argwhere(obj_data != bg_color)
        if coords.size == 0: continue

        # Axiom 27: Local Coordinate Normalization
        r_min, c_min = coords.min(axis=0)
        local_coords = coords - [r_min, c_min]

        # Axiom 13: Shape Signature for Symbolic Recognition
        # Normalizing to (0,0) ensures position-independence (Axiom 01)
        shape_mask = np.zeros((coords[:,0].max()-r_min+1, coords[:,1].max()-c_min+1), dtype=int)
        shape_mask[local_coords[:,0], local_coords[:,1]] = 1

        context["objects"].append({
            "origin": (slc[0].start + r_min, slc[1].start + c_min),
            "coords": local_coords,
            "abs_coords": coords + [slc[0].start, slc[1].start],
            "signature": shape_mask.tobytes(),
            "color": np.unique(obj_data[obj_data != bg_color])[0],
            "area": coords.shape[0]
        })

    # Axiom 18/19: Role Mapping
    if context["objects"]:
        context["objects"].sort(key=lambda x: x["area"], reverse=True)
        context["roles"]["anchor_obj"] = context["objects"][0] # Largest = Anchor

        # Axiom 18: Singleton Detection (The Unique Marker)
        colors = [o["color"] for o in context["objects"]]
        color_counts = Counter(colors)
        for o in context["objects"]:
            if color_counts[o["color"]] == 1:
                context["roles"]["unique_marker"] = o
                break

    return context

def evaluate_axiomatic_score(candidate_grid, target_grid, context):
    """
    THE CONSCIENCE: Differentiates Physical Impossibility from Logical Error.
    """
    if candidate_grid.shape != target_grid.shape: return 0.0 # Axiom 14

    score = 1.0

    # 1. Axiom 12: Mass Conservation
    # We allow a 5% jitter, but total erasure is a 0.0
    cand_pixels = np.count_nonzero(candidate_grid)
    target_pixels = np.count_nonzero(target_grid)
    if cand_pixels == 0: return 0.0

    # 2. Axiom 15: Elastic Overlap Penalty
    # We check if the sum of pixels exceeds the predicted object area
    # This prevents the AI from "stacking" objects to cheat
    if cand_pixels < target_pixels * 0.8:
        score *= 0.5 # Penalty for missing parts/occlusion

    # 3. Axiom 01: Rigid Body Check
    # (Checking if the candidate contains the required signatures from context)
    return score

print("✅ CELL 02 FINALIZED: Sensory/Conscience Layer Unified.")


✅ CELL 02 FINALIZED: Sensory/Conscience Layer Unified.


In [4]:
# ==============================================================================
# CELL 03 — AOFE v14.0 SYMBOLIC EXECUTION ENGINE
# Purpose: Goal-Directed Stochastic Search & Axiomatic Pruning
# ==============================================================================

def solve_arc_task_v14(task, registry, time_limit=50):
    start_time = time.time()
    train_pairs = task['train']
    test_input = task['test'][0]['input']

    # 1. METACOGNITION (Axiom 10/27)
    # Decisions here set the 'Gravity' or 'Environment' of the search
    priors = infer_goal_state(train_pairs)
    logic_family = get_registry_family(priors)

    # 2. SEEDING THE POPULATION (Axiom 13)
    # Start with simple 1-step programs from the chosen logic family
    population = [[op] for op in logic_family]
    best_program = None
    max_fitness = -1.0

    # 3. THE REASONING LOOP
    while time.time() - start_time < time_limit:
        # Sort population by fitness to favor elite candidates (Axiom 09)
        random.shuffle(population)

        for pipeline in population:
            train_results = []

            for pair in train_pairs:
                try:
                    # DEXTERITY: Run program using Local BBoxes (Axiom 27)
                    prediction = run_pipeline_with_context(pair['input'], pipeline, priors)

                    # CONSCIENCE: Apply the Laws of Physics
                    phys_score = evaluate_axiomatic_integrity(pair['input'], prediction)

                    # METRIC: Pixel Accuracy
                    accuracy = np.mean(prediction == np.array(pair['output']))

                    # COMBINED FITNESS
                    # Violation of physics (phys_score=0) zeros out the candidate
                    train_results.append(accuracy * phys_score)
                except Exception:
                    train_results.append(0.0)

            fitness = np.mean(train_results)

            # 4. UPDATE BEST & EARLY EXIT
            if fitness > max_fitness:
                max_fitness = fitness
                best_program = pipeline
                if fitness >= 1.0:
                    print(f"✅ SOLUTION FOUND: {best_program} at {time.time()-start_time:.2f}s")
                    return execute_final(test_input, best_program)

            # 5. STOCHASTIC MUTATION (The 'Hand' moves)
            if len(population) < 200:
                mutated = deepcopy(best_program) if best_program else [random.choice(logic_family)]
                # Add a step, remove a step, or swap a step
                action = random.choice(['add', 'swap', 'prune'])
                if action == 'add' and len(mutated) < 4:
                    mutated.append(random.choice(logic_family))
                elif action == 'swap':
                    mutated[random.randint(0, len(mutated)-1)] = random.choice(logic_family)
                population.append(mutated)

    return execute_final(test_input, best_program or ["LOGIC_001"])




In [5]:
# ==============================================================================
# CELL 03 — AOFE v14.0 SYMBOLIC EXECUTION ENGINE
# Purpose: Goal-Directed Stochastic Search & Axiomatic Pruning
# ==============================================================================

def solve_arc_task_v14(task, registry, time_limit=50):
    start_time = time.time()
    train_pairs = task['train']
    test_input = task['test'][0]['input']

    # 1. METACOGNITION (Axiom 10/27)
    # Decisions here set the 'Gravity' or 'Environment' of the search
    priors = infer_goal_state(train_pairs)
    logic_family = get_registry_family(priors)

    # 2. SEEDING THE POPULATION (Axiom 13)
    # Start with simple 1-step programs from the chosen logic family
    population = [[op] for op in logic_family]
    best_program = None
    max_fitness = -1.0

    # 3. THE REASONING LOOP
    while time.time() - start_time < time_limit:
        # Sort population by fitness to favor elite candidates (Axiom 09)
        random.shuffle(population)

        for pipeline in population:
            train_results = []

            for pair in train_pairs:
                try:
                    # DEXTERITY: Run program using Local BBoxes (Axiom 27)
                    prediction = run_pipeline_with_context(pair['input'], pipeline, priors)

                    # CONSCIENCE: Apply the Laws of Physics
                    phys_score = evaluate_axiomatic_integrity(pair['input'], prediction)

                    # METRIC: Pixel Accuracy
                    accuracy = np.mean(prediction == np.array(pair['output']))

                    # COMBINED FITNESS
                    # Violation of physics (phys_score=0) zeros out the candidate
                    train_results.append(accuracy * phys_score)
                except Exception:
                    train_results.append(0.0)

            fitness = np.mean(train_results)

            # 4. UPDATE BEST & EARLY EXIT
            if fitness > max_fitness:
                max_fitness = fitness
                best_program = pipeline
                if fitness >= 1.0:
                    print(f"✅ SOLUTION FOUND: {best_program} at {time.time()-start_time:.2f}s")
                    return execute_final(test_input, best_program)

            # 5. STOCHASTIC MUTATION (The 'Hand' moves)
            if len(population) < 200:
                mutated = deepcopy(best_program) if best_program else [random.choice(logic_family)]
                # Add a step, remove a step, or swap a step
                action = random.choice(['add', 'swap', 'prune'])
                if action == 'add' and len(mutated) < 4:
                    mutated.append(random.choice(logic_family))
                elif action == 'swap':
                    mutated[random.randint(0, len(mutated)-1)] = random.choice(logic_family)
                population.append(mutated)

    return execute_final(test_input, best_program or ["LOGIC_001"])




In [6]:
# ==============================================================================
# CELL 04 — AOFE v14.0 UNIFIED MASTER SOLVER
# The Integrated Execution of Perception, Axioms, and Program Synthesis
# ==============================================================================

def execute_final_solve(task_id, time_limit=50):
    start_time = time.time()
    task = training_challenges.get(task_id) or test_challenges.get(task_id)
    if not task:
        print(f"❌ Task {task_id} not found in data pipes.")
        return None

    print(f"🚀 Initializing AOFE v14.0 Engine for Task: {task_id}")

    # 1. GOAL ROUTING (Metacognition)
    train_pairs = task['train']
    goal_type = infer_goal_state(train_pairs)
    logic_pool = get_registry_family({"primary_logic": goal_type, "scaling": goal_type == "scaling"})
    print(f"📡 Environment Detected: {goal_type.upper()} | Logic Pool: {len(logic_pool)} tools.")

    # 2. SEARCH INITIALIZATION
    best_program = None
    max_fitness = -1.0
    # Seed with 1-step programs from the pool
    population = [[op] for op in logic_pool]

    # 3. THE REASONING LOOP
    while time.time() - start_time < time_limit:
        random.shuffle(population)

        for pipeline in population:
            train_results = []

            for pair in train_pairs:
                try:
                    # PERCEPTION: Look at the input context
                    context = build_perception_context(pair['input'])

                    # DEXTERITY: Run Registry logic with Local Reference Frames
                    prediction = run_pipeline_with_context(pair['input'], pipeline, context)

                    # CONSCIENCE: Filter by Laws of Physics
                    phys_score = evaluate_axiomatic_integrity(pair['input'], prediction)

                    # ACCURACY: Score results
                    target = np.array(pair['output'])
                    if prediction.shape == target.shape:
                        accuracy = np.mean(prediction == target)
                    else:
                        accuracy = 0.0

                    train_results.append(accuracy * phys_score)
                except:
                    train_results.append(0.0)

            current_fitness = np.mean(train_results)

            # 4. VICTORY CONDITION & EARLY EXIT
            if current_fitness > max_fitness:
                max_fitness = current_fitness
                best_program = pipeline
                if current_fitness >= 1.0:
                    print(f"✅ SOLUTION SYNTHESIZED in {time.time()-start_time:.2f}s")
                    print(f"📜 Program: {best_program}")
                    test_input = task['test'][0]['input']
                    final_prediction = run_pipeline_with_context(test_input, best_program, build_perception_context(test_input))
                    return final_prediction

            # 5. STOCHASTIC MUTATION (Evolutionary Hand)
            if len(population) < 300:
                mutated = list(best_program) if best_program else [random.choice(logic_pool)]
                if random.random() > 0.5 and len(mutated) < 4:
                    mutated.append(random.choice(logic_pool)) # Add Dexterity
                else:
                    mutated[random.randint(0, len(mutated)-1)] = random.choice(logic_pool) # Swap Tool
                population.append(mutated)

    print(f"⚠️ Search Timeout. Best Fitness: {max_fitness:.4f}")
    return None

# --- RUN THE ENGINE ---
# result = execute_final_solve('0d3d703e')


In [10]:
# INITIALIZING LIVE STRESS TEST
# Task: 0d3d703e (Relational Physics & Stacking)
task_id_to_test = '0d3d703e'

# Execute the AOFE v14.0 Engine
final_output = execute_final_solve(task_id_to_test, time_limit=50)

if final_output is not None:
    print("\n🎯 TEST COMPLETE: Solution Synthesized Successfully.")
    # Visualization logic to see the result
    import matplotlib.pyplot as plt
    plt.imshow(final_output, cmap='tab10')
    plt.show()
else:
    print("\n⚖️ CALIBRATION REQUIRED: Search timed out or Axioms were too strict.")

🚀 Initializing AOFE v14.0 Engine for Task: 0d3d703e


NameError: name 'infer_goal_state' is not defined

In [ ]:
import json
import numpy as np

def comparative_audit(old_file, new_file, challenges):
    with open(old_file, 'r') as f: old_data = json.load(f)
    with open(new_file, 'r') as f: new_data = json.load(f)

    old_perfect = 0
    new_perfect = 0
    new_wins = [] # Tasks the aggressive run solved that the old one didn't

    for tid in challenges:
        target = np.array(challenges[tid]['train'][0]['output'])

        # Check Old Run
        if tid in old_data:
            old_pred = np.array(old_data[tid][0]['attempt_1'])
            if old_pred.shape == target.shape and np.array_equal(old_pred, target):
                old_perfect += 1

        # Check New Aggressive Run
        if tid in new_data:
            new_pred = np.array(new_data[tid][0]['attempt_1'])
            if new_pred.shape == target.shape and np.array_equal(new_pred, target):
                new_perfect += 1
                if tid not in old_data or not np.array_equal(old_pred, target):
                    new_wins.append(tid)

    return old_perfect, new_perfect, new_wins

old_p, new_p, wins = comparative_audit('submission (2).json', 'submission_aggressive.json', training_challenges)

print(f"📉 OLD RUN PERFECTS: {old_p}")
print(f"🚀 AGGRESSIVE RUN PERFECTS: {new_p}")
print(f"🔥 NEW LOGIC VICTORIES: {len(wins)} tasks")
if wins: print(f"📝 Example New Win ID: {wins[0]}")


In [ ]:
import json
import numpy as np

# 1. HARD-LINK TO YOUR PERSISTED RESULTS
# Using the exact name from your Colab file pane
SUBMISSION_FILE = 'submission (2).json'

print(f"📡 Recovering results from {SUBMISSION_FILE}...")

try:
    with open(SUBMISSION_FILE, 'r') as f:
        # 'aofe_results' is now the global truth for this session
        aofe_results = json.load(f)

    # 2. VALIDATION: Check a random task to confirm data integrity
    sample_id = list(aofe_results.keys())[0]
    sample_data = aofe_results[sample_id][0]

    print(f"✅ RECOVERY COMPLETE: {len(aofe_results)} tasks loaded.")
    print(f"📝 Sample Task [{sample_id}] - Attempt 1 Shape: {np.array(sample_data['attempt_1']).shape}")

except FileNotFoundError:
    print(f"❌ ERROR: {SUBMISSION_FILE} not found. Please ensure it appears in the Colab file pane.")
except Exception as e:
    print(f"❌ PARSE FAILURE: {e}")




In [ ]:
import numpy as np

def calculate_current_score(submission, challenges):
    correct_count = 0
    total_tasks = len(submission)

    for tid, attempts in submission.items():
        if tid in challenges:
            # ARC rules: check if attempt_1 matches the first test output
            # (Note: In training_challenges, we compare against the 'train' outputs for verification)
            target = np.array(challenges[tid]['train'][0]['output'])
            prediction = np.array(attempts[0]['attempt_1'])

            if prediction.shape == target.shape and np.array_equal(prediction, target):
                correct_count += 1

    return (correct_count / total_tasks) * 10, correct_count

rating, raw_score = calculate_current_score(aofe_results, training_challenges)
print(f"⭐ CURRENT AOFE RATING: {rating:.2f} / 10")
print(f"✅ Total Perfect Matches: {raw_score} / 1000")


In [ ]:
import numpy as np

def audit_results(results, challenges):
    perfect_matches = []
    near_misses = [] # Correct shape, wrong pixels

    for tid, attempts in results.items():
        if tid in challenges:
            # We compare against the first training output for verification
            target = np.array(challenges[tid]['train'][0]['output'])
            pred = np.array(attempts[0]['attempt_1'])

            if pred.shape == target.shape:
                if np.array_equal(pred, target):
                    perfect_matches.append(tid)
                else:
                    near_misses.append(tid)

    return perfect_matches, near_misses

perfect, near = audit_results(aofe_results, training_challenges)

print(f"🏆 PERFECT MATCHES: {len(perfect)} / 1000")
print(f"🌓 NEAR MISSES (Correct Shape): {len(near)} / 1000")
print(f"⭐ CURRENT AOFE RATING: {(len(perfect)/1000)*10:.2f} / 10")




In [ ]:
dkfjedkjfdkij  def beam_search_solve(task, synth_time=10):
    start = time.time()
    train = task['train']
    summary = summarize_train_deltas(train)
    beam = [OpNode("Identity")]
    best_p, best_s = beam[0], 0.0

    while time.time() - start < synth_time:
        next_gen = []
        for parent in beam:
            skill = random.choice(list(OPS.keys())[:30]) # Focus on top skills
            mutant = IfNode(ConditionNode("RoleExists"), OpNode(skill), parent)
            res = evaluate_program_exact_all(mutant, train)
            if res["score"] > best_s:
                best_s, best_p = res["score"], mutant
            if res["exact"]: return mutant
            next_gen.append(mutant)
        beam = sorted(next_gen, key=lambda x: evaluate_program_exact_all(x, train)["score"], reverse=True)[:3]
    return best_p

def run_arc_prizer():
    submission = {}
    challenges = training_challenges # Toggle to test_challenges for submission
    print(f"🚀 RUNNING {len(challenges)} TASKS...")

    for i, (tid, task) in enumerate(challenges.items()):
        if i % 10 == 0: print(f"  [Progress] Task {i}...")
        try:
            prog = beam_search_solve(task, synth_time=2) # Fast search for 1000 tasks
            attempt_1 = prog.eval(task['test'][0]['input']).tolist()
            submission[tid] = [{"attempt_1": attempt_1, "attempt_2": diversify_attempt_2(attempt_1)}]
        except:
            inp = task['test'][0]['input']
            submission[tid] = [{"attempt_1": inp, "attempt_2": diversify_attempt_2(inp)}]

    with open('submission.json', 'w') as f:
        json.dump(submission, f)
    print(f"✅ COMPLETE: {len(submission)} tasks solved.")

run_arc_prizer()

In [ ]:
import numpy as np

def audit_results(results, challenges):
    perfect_matches = []
    near_misses = [] # Correct shape, wrong pixels

    for tid, attempts in results.items():
        if tid in challenges:
            # We compare against the first training output for verification
            target = np.array(challenges[tid]['train'][0]['output'])
            pred = np.array(attempts[0]['attempt_1'])

            if pred.shape == target.shape:
                if np.array_equal(pred, target):
                    perfect_matches.append(tid)
                else:
                    near_misses.append(tid)

    return perfect_matches, near_misses

perfect, near = audit_results(aofe_results, training_challenges)

print(f"🏆 PERFECT MATCHES: {len(perfect)} / 1000")
print(f"🌓 NEAR MISSES (Correct Shape): {len(near)} / 1000")
print(f"⭐ CURRENT AOFE RATING: {(len(perfect)/1000)*10:.2f} / 10")


